# 01 — Azure DevOps Pipeline YAML Walkthrough

Companion notebook to `02-azure-devops-pipelines-yaml.md`. This notebook is fully offline: no real Azure DevOps project, Azure subscription, or network access is needed. It parses a **sample Azure Pipelines YAML string** (shaped like the worked example in the chapter, deploying a containerized FastAPI app such as the document-uploader service from course 05 — FastAPI, SQLAlchemy ORM, RBAC via MSAL/OAuth 2.0) using the `yaml` module, validates that the structure has the required top-level keys, and pretty-prints the stage -> job -> step tree.

**Setup:** this notebook uses [PyYAML](https://pypi.org/project/pyyaml/). If it isn't already installed in your environment, run:

```bash
pip install pyyaml
```

In [1]:
import yaml
from pprint import pprint

## 1. A sample Azure DevOps pipeline

This mirrors the worked example from `02-azure-devops-pipelines-yaml.md`: a `Build` stage (install deps, lint, test, build/push a Docker image) followed by a `DeployDev` stage that deploys to an Azure App Service slot and runs a post-deploy smoke test (the same smoke-test pattern built out in notebook 03).

In [2]:
sample_pipeline_yaml = """
trigger:
  branches:
    include: [main]

pr:
  branches:
    include: [main]

variables:
  - group: uploader-svc-vars
  - name: imageRepository
    value: 'document-uploader'

stages:
  - stage: Build
    jobs:
      - job: BuildTestScan
        pool:
          vmImage: 'ubuntu-latest'
        steps:
          - script: pip install -r requirements.txt && flake8 app/
            displayName: 'Install deps + lint'
          - script: pytest --junitxml=test-results.xml
            displayName: 'Unit + integration tests'
          - task: Docker@2
            displayName: 'Build and push image'

  - stage: DeployDev
    dependsOn: Build
    jobs:
      - deployment: DeployToDev
        environment: 'uploader-dev'
        pool:
          vmImage: 'ubuntu-latest'
        strategy:
          runOnce:
            deploy:
              steps:
                - task: AzureWebApp@1
                  displayName: 'Deploy to Azure App Service (dev)'
                - script: python scripts/smoke_test.py --url https://document-uploader-dev.azurewebsites.net
                  displayName: 'Post-deploy smoke test'
"""

print(sample_pipeline_yaml)


trigger:
  branches:
    include: [main]

pr:
  branches:
    include: [main]

variables:
  - group: uploader-svc-vars
  - name: imageRepository
    value: 'document-uploader'

stages:
  - stage: Build
    jobs:
      - job: BuildTestScan
        pool:
          vmImage: 'ubuntu-latest'
        steps:
          - script: pip install -r requirements.txt && flake8 app/
            displayName: 'Install deps + lint'
          - script: pytest --junitxml=test-results.xml
            displayName: 'Unit + integration tests'
          - task: Docker@2
            displayName: 'Build and push image'

  - stage: DeployDev
    dependsOn: Build
    jobs:
      - deployment: DeployToDev
        environment: 'uploader-dev'
        pool:
          vmImage: 'ubuntu-latest'
        strategy:
          runOnce:
            deploy:
              steps:
                - task: AzureWebApp@1
                  displayName: 'Deploy to Azure App Service (dev)'
                - script: python scripts/smoke_

## 2. Parse it

`yaml.safe_load` turns the YAML text into plain Python `dict`/`list` structures — this is exactly what Azure DevOps itself does internally before scheduling stages, jobs, and steps onto agents.

In [3]:
pipeline = yaml.safe_load(sample_pipeline_yaml)
print(type(pipeline))
pprint(pipeline, sort_dicts=False)

<class 'dict'>
{'trigger': {'branches': {'include': ['main']}},
 'pr': {'branches': {'include': ['main']}},
 'variables': [{'group': 'uploader-svc-vars'},
               {'name': 'imageRepository', 'value': 'document-uploader'}],
 'stages': [{'stage': 'Build',
             'jobs': [{'job': 'BuildTestScan',
                       'pool': {'vmImage': 'ubuntu-latest'},
                       'steps': [{'script': 'pip install -r requirements.txt '
                                            '&& flake8 app/',
                                  'displayName': 'Install deps + lint'},
                                 {'script': 'pytest '
                                            '--junitxml=test-results.xml',
                                  'displayName': 'Unit + integration tests'},
                                 {'task': 'Docker@2',
                                  'displayName': 'Build and push image'}]}]},
            {'stage': 'DeployDev',
             'dependsOn': 'Build',
        

## 3. Validate required top-level structure

A minimally well-formed pipeline needs a `stages` list, each stage needs `jobs`, and each job needs `steps` (either directly, or nested under `strategy.runOnce.deploy.steps` for `deployment` jobs targeting an Environment — see chapter 02's discussion of deployment jobs vs. plain jobs). `validate_pipeline` below walks the structure and reports anything missing, the way a linting stage in CI might catch a malformed pipeline file before it ever reaches Azure DevOps.

In [4]:
def validate_pipeline(pipeline):
    """Return a list of validation issues; an empty list means the structure looks valid."""
    issues = []
    if not isinstance(pipeline, dict):
        return ["Top-level pipeline document is not a mapping."]
    if "stages" not in pipeline:
        issues.append("Missing top-level 'stages' key.")
        return issues
    stages = pipeline["stages"]
    if not isinstance(stages, list) or not stages:
        issues.append("'stages' must be a non-empty list.")
        return issues
    for i, stage in enumerate(stages):
        stage_name = stage.get("stage", f"<unnamed stage #{i}>")
        jobs = stage.get("jobs")
        if not jobs:
            issues.append(f"Stage '{stage_name}' has no 'jobs'.")
            continue
        for job in jobs:
            job_name = job.get("job") or job.get("deployment") or "<unnamed job>"
            steps = job.get("steps")
            if steps is None and "strategy" in job:
                steps = (
                    job.get("strategy", {})
                    .get("runOnce", {})
                    .get("deploy", {})
                    .get("steps")
                )
            if not steps:
                issues.append(f"Job '{job_name}' in stage '{stage_name}' has no 'steps'.")
    return issues


issues = validate_pipeline(pipeline)
print("Validation issues:", issues if issues else "None - pipeline structure looks valid")

Validation issues: None - pipeline structure looks valid


In [5]:
# Sanity check: validation should catch a pipeline with a job that has no steps
broken_pipeline_yaml = """
stages:
  - stage: Build
    jobs:
      - job: BuildOnly
        pool:
          vmImage: 'ubuntu-latest'
"""
broken = yaml.safe_load(broken_pipeline_yaml)
print(validate_pipeline(broken))

["Job 'BuildOnly' in stage 'Build' has no 'steps'."]


## 4. Pretty-print the stage -> job -> step tree

This is the same mental model from the chapter: **stage = phase + gate boundary, job = machine + parallelism boundary, step = command**. Printing it as an indented tree makes that hierarchy visually obvious, which is useful both for debugging a real pipeline and for explaining the YAML structure out loud in an interview.

In [6]:
def print_pipeline_tree(pipeline):
    lines = []
    for stage in pipeline.get("stages", []):
        lines.append(f"Stage: {stage.get('stage')}")
        for job in stage.get("jobs", []):
            job_name = job.get("job") or job.get("deployment")
            lines.append(f"  Job: {job_name}")
            steps = job.get("steps")
            if steps is None:
                steps = job.get("strategy", {}).get("runOnce", {}).get("deploy", {}).get("steps", [])
            for step in steps or []:
                label = step.get("displayName") or step.get("script") or step.get("task") or "<step>"
                lines.append(f"    Step: {label}")
    return "\n".join(lines)


print(print_pipeline_tree(pipeline))

Stage: Build
  Job: BuildTestScan
    Step: Install deps + lint
    Step: Unit + integration tests
    Step: Build and push image
Stage: DeployDev
  Job: DeployToDev
    Step: Deploy to Azure App Service (dev)
    Step: Post-deploy smoke test


## Takeaway

Being able to load a pipeline YAML file and mechanically answer "what stages exist, what runs in each job, in what order" is exactly the skill being tested when an interviewer hands you an unfamiliar `azure-pipelines.yml` and asks you to walk through what it does — see `99-Interview-QA.md` Q7 for the narrated version of this same walkthrough.